In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY is not set")
print("OPENAI_API_KEY is set")


OPENAI_API_KEY is set


## PART 1 — Text-to-Math Agent Overview

**Task 1: Conceptual Questions**
Answer briefly:

1. What is a Text-to-Math problem? → a word problem in natural language that needs math (arithmetic, %, algebra) to solve. e.g. "A shirt costs 800 and is 20% off — whats the final price?"
2. Why agents are useful for math reasoning? → LLMs alone often mess up multi-step calc. an agent can *plan* steps and call a calculator tool for exact numbers instead of guessing.
3. Difference between normal LLM response vs agent-based reasoning → normal LLM: one-shot answer (may hallucinate math). agent: think → maybe use tool → observe result → final answer (more reliable on numbers).


## PART 2 — Build Text-to-Math Agent

**Task 2: Build Text-to-Math Agent**
1. LangChain agent + math/calculator tool + LLM
2. Understand word problems → break into steps → calculate → final answer
3. Test arithmetic, percentage, simple algebra


In [2]:
from typing import Literal
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
import ast
import operator as op

In [ ]:
_OPS = {
    ast.Add: op.add,
    ast.Sub: op.sub,
    ast.Mult: op.mul,
    ast.Div: op.truediv,
    ast.Pow: op.pow,
    ast.USub: op.neg,
    ast.Mod: op.mod,
}

def _eval_node(node):
    if isinstance(node, ast.Expression):
        return _eval_node(node.body)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.Num): 
        return node.n
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval_node(node.left), _eval_node(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval_node(node.operand))
    raise ValueError(f"unsupported expression: {ast.dump(node)}")


In [4]:
@tool
def calculator(expression: str) -> float:
    """Evaluate a math expression like '2+(3*9)' or '800*0.8'.
    Use this for exact arithmetic instead of guessing.
    """
    tree = ast.parse(expression.replace("^", "**"), mode="eval")
    return float(_eval_node(tree))

In [5]:
print(calculator.invoke({"expression": "2+(3*9)"}))
print(calculator.invoke({"expression": "800*(1-0.2)"}))

29.0
640.0


/var/folders/3k/2r4fckl56zxdspgzm1tw5pdw0000gn/T/ipykernel_48894/2721198973.py:16: DeprecationWarning: ast.Num is deprecated and will be removed in Python 3.14; use ast.Constant instead
  if isinstance(node, ast.Num):  # py<3.8 style, just in case


In [7]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

system_prompt = """You are a Text-to-Math agent.
- Read the word problem carefully.
- Break it into clear calculation steps.
- Use the calculator tool for exact arithmetic / percentages / simple algebra numbers.
- Show brief reasoning, then give the Final Answer clearly.
- If info is missing, ask a clarifying question.
"""

math_agent = create_agent(
    llm,
    tools=[calculator],
    system_prompt=system_prompt,
)

def ask_math(question: str) -> str:
    result = math_agent.invoke({"messages": [("user", question)]})
    return result["messages"][-1].content


### Test cases
- Arithmetic word problem
- Percentage problem
- Simple algebra


In [8]:
tests = [
    # arithmetic
    "Ravi buys 3 notebooks for 45 rupees each and 2 pens for 20 rupees each. What is the total cost?",
    # percentage
    "A shirt costs 800 rupees and is on 20% discount. What is the final price?",
    # simple algebra-ish
    "Solve for x: 2x + 5 = 17. What is x?",
]

for q in tests:
    print("Q:", q)
    print("A:", ask_math(q))
    print("-" * 80)


Q: Ravi buys 3 notebooks for 45 rupees each and 2 pens for 20 rupees each. What is the total cost?


/var/folders/3k/2r4fckl56zxdspgzm1tw5pdw0000gn/T/ipykernel_48894/2721198973.py:16: DeprecationWarning: ast.Num is deprecated and will be removed in Python 3.14; use ast.Constant instead
  if isinstance(node, ast.Num):  # py<3.8 style, just in case


A: To find the total cost, we need to calculate the cost of the notebooks and the pens separately and then add them together.

1. Cost of notebooks: \(3 \text{ notebooks} \times 45 \text{ rupees each} = 135 \text{ rupees}\)
2. Cost of pens: \(2 \text{ pens} \times 20 \text{ rupees each} = 40 \text{ rupees}\)

Now, we add the two costs together:
- Total cost = \(135 \text{ rupees} + 40 \text{ rupees} = 175 \text{ rupees}\)

Final Answer: 175 rupees.
--------------------------------------------------------------------------------
Q: A shirt costs 800 rupees and is on 20% discount. What is the final price?


/var/folders/3k/2r4fckl56zxdspgzm1tw5pdw0000gn/T/ipykernel_48894/2721198973.py:16: DeprecationWarning: ast.Num is deprecated and will be removed in Python 3.14; use ast.Constant instead
  if isinstance(node, ast.Num):  # py<3.8 style, just in case


A: To find the final price of the shirt after a 20% discount:

1. Calculate the discount amount:
   - Discount = 20% of 800 rupees = 160 rupees.

2. Subtract the discount from the original price:
   - Final Price = 800 rupees - 160 rupees = 640 rupees.

Final Answer: The final price of the shirt is 640 rupees.
--------------------------------------------------------------------------------
Q: Solve for x: 2x + 5 = 17. What is x?


/var/folders/3k/2r4fckl56zxdspgzm1tw5pdw0000gn/T/ipykernel_48894/2721198973.py:16: DeprecationWarning: ast.Num is deprecated and will be removed in Python 3.14; use ast.Constant instead
  if isinstance(node, ast.Num):  # py<3.8 style, just in case


A: To solve for \( x \) in the equation \( 2x + 5 = 17 \), we can follow these steps:

1. Subtract 5 from both sides: 
   \[
   2x = 17 - 5
   \]
2. Divide both sides by 2:
   \[
   x = \frac{12}{2}
   \]

Calculating this gives us \( x = 6 \).

Final Answer: \( x = 6 \)
--------------------------------------------------------------------------------


## PART 3 — Streamlit App with Session State

**Task 3: Session State for Application**
1. Streamlit chat UI (`app.py`)
2. `st.session_state` keeps previous Q&A
3. math context preserved across turns (chat history messages)

Run:
```bash
cd genai/langchain/assignment-35
streamlit run app.py
```


## Observations (short)

1. Calculator tool fixes dumb arithmetic mistakes LLMs make on word problems.
2. Agent is overkill for "2+2", but useful when the problem needs parsing + calc.
3. Session state matters in Streamlit because the script reruns every click — without it, history vanishes.
